<a href="https://colab.research.google.com/github/CplSwofford/Signal_Systems/blob/main/ISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# needed
import os
import re
import soundfile as sf
from IPython.display import Audio
from IPython.display import display
import numpy as np
# recommended ...
from scipy import signal
from scipy.io import wavfile
from scipy.fft import fft, ifft, fftfreq
import scipy.io
import matplotlib.pyplot as plt


In [ ]:
# read data
login = "239468"
zip_file = "known" + ".zip"
for name in ("known.zip", "valid.zip", login + ".zip"):
  file = "https://www.fit.vut.cz/study/course/ISS/public/proj2025-26/" + name
  !rm $name
  !wget $file
  !unzip -o -q $name

--2025-11-08 09:28:47--  https://www.fit.vut.cz/study/course/ISS/public/proj2025-26/known.zip
Resolving www.fit.vut.cz (www.fit.vut.cz)... 147.229.9.65, 2001:67c:1220:809::93e5:941
Connecting to www.fit.vut.cz (www.fit.vut.cz)|147.229.9.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 210919709 (201M) [application/zip]
Saving to: ‘known.zip’

known.zip           100%[===================>] 201.15M  24.7MB/s    in 20s     

2025-11-08 09:29:08 (10.2 MB/s) - ‘known.zip’ saved [210919709/210919709]

--2025-11-08 09:29:11--  https://www.fit.vut.cz/study/course/ISS/public/proj2025-26/valid.zip
Resolving www.fit.vut.cz (www.fit.vut.cz)... 147.229.9.65, 2001:67c:1220:809::93e5:941
Connecting to www.fit.vut.cz (www.fit.vut.cz)|147.229.9.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7608498 (7.3M) [application/zip]
Saving to: ‘valid.zip’

valid.zip           100%[===================>]   7.26M  1.05MB/s    in 20s     

2025-11-08 09:29:32

In [ ]:
# load the known data - the function returns a big matrix with all the signals
def load_data (S, dirname, count, no_samples):
  ii = 0
  for one in np.arange(count):
    S[ii], Fs = sf.read(dirname + "/" + str(one) + ".wav")
    ii = ii+1

Fs = 16000
# load known data
N_known = 706; duration_known = 10; no_samples_known = Fs * duration_known
known_signals=np.zeros([N_known, no_samples_known]); load_data(known_signals, "known", N_known, no_samples_known)
#display(Audio(known_signals[45], rate=Fs))

# load validation data
N_valid = 50; duration_valid = 5; no_samples_valid = Fs * duration_valid
valid_signals= np.zeros([N_valid, no_samples_valid]); load_data(valid_signals, "valid", N_valid, no_samples_valid)
#display(Audio(valid_signals[45], rate=Fs))

In [38]:
# YOUR CODE COMES HERE ! Your task is to produce a matrix 50 x 706 with similarity measures

# COUNT number of sound clips
def sound_clips(folder):
    return sum(len(files) for _, _, files in os.walk(folder))


# GET number of sound clips
known_snound_clips_num = sound_clips("known")   #706
valid_snound_clips_num = sound_clips("valid")   #51
#print(known_snound_clips_num)
#print(valid_snound_clips_num)


# EMPTY Matrix - for saving comparison results
emptyMatrix_Valid_to_Known = np.zeros((valid_snound_clips_num, known_snound_clips_num), dtype=int)
#print(emptyMatrix_50Valid_to_706known.shape)



# GET info about
# sampling frequency = _FS
# number of samples = _samples
known_Fs, known_samples = wavfile.read(f"known/{0}.wav")    #16000, #160000
known_sample_len = known_samples.shape[0]
#print(known_Fs, known_samples.shape, known_samples.dtype)
#print(known_samples[:10])

valid_Fs, valid_samples = wavfile.read(f"valid/{0}.wav")    #16000, #80000
valid_sample_len = valid_samples.shape[0]
#print(valid_Fs, valid_samples.shape, valid_samples.dtype)
#print(valid_samples[:10])


# EMPTY Vectors - for saving normalized values
valid_vec = np.zeros((valid_snound_clips_num, valid_sample_len), dtype=np.float32)
known_vec = np.zeros((known_snound_clips_num, known_sample_len), dtype=np.float32)
#print(known_vec[:10])
#print(known_vec.shape)


# NORMALIAUTION function: from −32768 -> +32767  to -1 -> 1
def int16_to_unit_closed(x):
    xf = x.astype(np.float32)
    return np.where(xf < 0, xf / 32768.0, xf / 32767.0).astype(np.float32)


for i_v in range(valid_snound_clips_num-1):
    _, v_s = wavfile.read(f"valid/{i_v}.wav")

    """
    if getattr(v_s, "ndim", 1) == 2:     # stereo -> mono (průměr)
        v_s = v_s.mean(axis=1)
    """
    valid_vec[i_v, :] = int16_to_unit_closed(v_s)


for i_k in range(known_snound_clips_num):
    _, k_s = wavfile.read(f"known/{i_k}.wav")

    """
    if getattr(k_s, "ndim", 1) == 2:
        k_s = k_s.mean(axis=1)
    """
    known_vec[i_k, :] = int16_to_unit_closed(k_s)




"""
#TEMP - DELETE
Fs, x = wavfile.read("TryToDownloadOtherDataset/14.wav")  # x je NumPy pole: int16/int32/float32
print(Fs, x.shape, x.dtype)
print(x[:20])

Fs, x = wavfile.read("known/14.wav")  # x je NumPy pole: int16/int32/float32
print(Fs, x.shape, x.dtype)
print(x[:20])
"""

"""
i = 14

Fs, x = wavfile.read(f"known/{i}.wav")
print(Fs, x.shape, x.dtype)
print(x[:20])

t = np.arange(len(x)) / Fs     # časová osa v sekundách
plt.plot(t[:1000], x[:1000])   # prvních 1000 vzorků (~0.0625 s)
plt.xlabel("čas [s]")
plt.ylabel("amplituda")
plt.title("Časový průběh WAV signálu")
plt.grid()
plt.show()
"""
"""
def compute_similarity_matrix(N_valid, N_known):
  # complete stupid solution just for testing the evaluation ...
  # the full version will probably have more arguments !
  similarities = np.random.uniform( 0, 1, (N_valid, N_known))
  return similarities
"""

'\ndef compute_similarity_matrix(N_valid, N_known):\n  # complete stupid solution just for testing the evaluation ...\n  # the full version will probably have more arguments !\n  similarities = np.random.uniform( 0, 1, (N_valid, N_known))\n  return similarities\n'

In [ ]:
# evaluation - the function produces Top-1 and Top-5 accuracy on validation data
def eval(scores, key):
  indices = np.flip(np.argsort(scores), axis=-1) # we want highest to lowest ...
  #print(scores[0,key[0]], key[0], indices)
  top1acc = np.sum(key == indices[:,0]) / indices.shape[0]
  top5acc = 0
  for ii in range(5):
    top5acc += np.sum(key == indices[:,ii])
  top5acc /=  indices.shape[0]
  return top1acc, top5acc

key = np.loadtxt("valid/key.txt", delimiter = ',', usecols=(1), dtype ='int')